In [ ]:
dev_id = "20260723-032552-e46d2b"
val_id = "20260723-032553-e79bce"

In [ ]:
# trainer config, same as the one used for the human data runs, bar the dataset name
cfg = {
    "model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
    "neg_ratio": None, "seed": 0, "dataset": "synthetic",
}

SYNTH_VERSION = "v14"

GEN_CFG  = {"model": "gpt-oss-120b", "temperature": 0.7,
            "reasoning_effort": "low", "max_output_tokens": 3000}
SPEC_CFG = {"n": 2000, "seed": 0}


In [ ]:
import os
from openai import OpenAI

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)


In [ ]:
# 1. setup
import importlib, json, sys
import ddi.vocab, ddi.prompt, ddi.resolve, ddi.synth, ddi.gates
for m in (ddi.vocab, ddi.prompt, ddi.resolve, ddi.synth, ddi.gates):
    importlib.reload(m)

from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, render_v14, make_v14_sample_fn, v14_fingerprint
from ddi.resolve import v14_sample_to_instances, generation_records, separation_report
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates

GEN_ID = "v14-smoke-8"          # bump on every config change: generate_raw resumes by index
MODEL, API, EFFORT, MAX_TOKENS = "gpt-oss-120b", "responses", "low", 3000

vocab = build_vocab()
print(f"{len(vocab.drugs)} drugs, {len(vocab.groups)} groups")

In [ ]:
# 2. render and read before spending calls
specs = make_v14_specs(300, vocab=vocab, seed=1, composition="prior")
for s in specs:
    print(render_v14(s), "\n" + "-" * 60)

from collections import Counter
print(Counter(s["asserts"][0]["label"] if s["asserts"] else "NONE" for s in specs))
print(Counter(len(s["entities"]) for s in specs))

In [ ]:
# 3. generate
import time
t0 = time.time()
sample_fn = make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT, api=API, max_output_tokens=MAX_TOKENS)
generate_raw(specs, sample_fn, gen_id=GEN_ID, max_workers=8)
print(f"{(time.time() - t0) / len(specs):.1f}s per sentence")   

In [ ]:
# 4. read the sentences against their specs
for line in (RAW / f"{GEN_ID}.jsonl").read_text().splitlines():
    if not line:
        continue
    rec = json.loads(line)
    if rec.get("error"):
        print("ERR", rec["error"][:120], "\n")
        continue
    spec = rec["spec"]
    lab = spec["asserts"][0]["label"] if spec["asserts"] else "NONE"
    print(f"[{lab} {spec['register']} n={len(spec['entities'])}]")
    print(" ", rec["sample"]["sentence"])
    for a in spec["asserts"]:
        print(f"   asserted: {a['between']}  {a['content']}")
    print()

In [ ]:
from collections import Counter, defaultdict
by_lab = defaultdict(Counter)
for s in specs:
    lab = s["asserts"][0]["label"] if s["asserts"] else "NONE"
    by_lab[lab][s["style"]["length"]] += 1
for lab, c in by_lab.items():
    n = sum(c.values())
    print(f"{lab:10s} " + "  ".join(f"{k.split()[0]}:{v/n:.2f}" for k, v in c.items()))

In [ ]:
import statistics
lens = defaultdict(list)
from ddi.resolve import generation_records
recs = generation_records(GEN_ID)


import statistics
from collections import defaultdict

lens = defaultdict(list)
by_req = defaultdict(list)
for r in recs:
    n = len(r["sentence"].split())
    lens["POS" if r["label"] != "NONE" else "NONE"].append(n)
    if r.get("length_requested"):
        by_req[r["length_requested"].split()[0]].append(n)

for k, v in lens.items():
    print(f"{k:5s} median {statistics.median(v):>3.0f}  n={len(v)}")
print()
for k, v in sorted(by_req.items()):
    print(f"{k:6s} median {statistics.median(v):>3.0f}  n={len(v)}")

In [ ]:
# 5. resolve and gate
import importlib, ddi.synth, ddi.resolve, ddi.prompt, ddi.gates
importlib.reload(ddi.synth)
importlib.reload(ddi.resolve)      # after synth, so it picks up the new Rejected
importlib.reload(ddi.prompt)
importlib.reload(ddi.gates)
from ddi.resolve import v14_sample_to_instances
from ddi.synth import build_dataset_from_raw

did, stats = build_dataset_from_raw(
    GEN_ID, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "composition": "prior",
               "model": MODEL, "reasoning_effort": EFFORT},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])

inst, _ = load_dataset(did)
recs = generation_records(GEN_ID)
gates.report(inst, records=recs, strict=False)
separation_report(recs)

for r in inst[:15]:
    print(f"{r['label']:10s} {r['text']}")

In [ ]:
from ddi.data import build_human
from ddi import gates

train, dev, val = build_human()
gates.report(train, strict=False)

In [ ]:
import random
rng = random.Random(3)
sents = sorted({r["sent_id"] for r in train})
keep = set(rng.sample(sents, 282))          # same sentence count as the synth run
sub = [r for r in train if r["sent_id"] in keep]
print(len(sub), "instances")
print(gates.shortcut_probe(sub)[0])

In [ ]:
import statistics
def probe_repeated(instances, n_seeds=10):
    xs = [gates.shortcut_probe(instances, seed=s)[0] for s in range(n_seeds)]
    return statistics.median(xs), statistics.stdev(xs), min(xs), max(xs)

In [ ]:
print("human-sub", *probe_repeated(sub))
print("synth", *probe_repeated(inst))

In [ ]:
# 6. rejects, which are the prompt-iteration diagnostic
p = RAW / f"{GEN_ID}.rejects.jsonl"
if p.exists():
    for line in p.read_text().splitlines()[:20]:
        r = json.loads(line)
        print(r["reason"])
        print("  ", (r["sample"] or {}).get("sentence", "")[:160], "\n")

In [ ]:
import importlib, ddi.prompt, ddi.resolve, ddi.gates
for m in (ddi.prompt, ddi.resolve, ddi.gates): importlib.reload(m)
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates
import json

GEN = "v14-check-1"
specs = make_v14_specs(150, vocab=vocab, seed=0)
generate_raw(specs, make_v14_sample_fn(client, reasoning_effort="low"),
             gen_id=GEN, max_workers=16)

# the three things being checked
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    r = json.loads(line)
    if r.get("error"): continue
    s, txt = r["spec"], r["sample"]["sentence"]
    lab = s["asserts"][0]["label"] if s["asserts"] else "NONE"
    bad = []
    if lab in ("INT","MECHANISM","EFFECT") and "recommend" in txt.lower(): bad.append("RECOMMEND")
    if "given alone" in txt or "relative to" in txt: bad.append("RELATIVE")
    if "evaluated separately" in txt: bad.append("R3")
    if bad: print(f"[{'/'.join(bad)} {lab}] {txt}\n")

did, stats = build_dataset_from_raw(GEN, resolver=v14_sample_to_instances,
    mode="markers", generator={"prompt_sha": v14_fingerprint()},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)
gates.report(inst, records=generation_records(GEN), strict=False)

In [ ]:
from ddi.data import build_human
from ddi.manifest import load_dataset

train, dev, val = build_human()
v14, _ = load_dataset("20260807-123340-ff79db")

In [ ]:
import re, statistics
from collections import Counter

ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    """text within `chars` either side of each entity span"""
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start()-chars):m.end()+chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        hit = bool(ROLE.search(near_marker(r["text"])))
        c[(r["label"] != "NONE", hit)] += 1
    pos_rate = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg_rate = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:8s} role language near marker: POS {pos_rate:.3f}  NONE {neg_rate:.3f}"
          f"  ratio {neg_rate / max(pos_rate, 1e-9):.2f}")

role_adjacency(v14, "v14")
role_adjacency(train, "human")

In [ ]:
# P_ROLE=0.2 smoke test
# Change ONE thing in ddi/prompt.py before running: P_ROLE = 0.55 -> 0.2
# Everything else stays as it was for v14-full-2, so the comparison is single-variable.
import os
from openai import OpenAI
import importlib, json, re
from collections import Counter

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, make_v14_sample_fn, v14_fingerprint, P_ROLE
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates

GEN = "v14-lowrole-check"
V14_ID = "20260807-123340-ff79db"
MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

print(f"P_ROLE = {P_ROLE}  (must be 0.2)")
print(f"prompt sha {v14_fingerprint()}")

vocab = build_vocab()
train, dev, val = build_human()
v14, _ = load_dataset(V14_ID)


# ---- generate -------------------------------------------------------------
specs = make_v14_specs(300, vocab=vocab, seed=0)

n_roled = sum(len(s["roles"]) for s in specs)
n_np = sum(len([e for e in s["entities"]
                if e["key"] not in {k for a in s["asserts"] for k in a["between"]}])
           for s in specs)
print(f"specs: {n_roled}/{n_np} non-participants carry a role "
      f"({n_roled / max(n_np, 1):.2f})")

generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "p_role": P_ROLE,
               "model": MODEL, "reasoning_effort": EFFORT,
               "note": "P_ROLE ablation against v14-full-2"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)


# ---- manipulation check: did the cue actually move? -----------------------
ROLE = re.compile(
    r"\b(previous(ly)?|earlier|discontinu\w+|stopped|withdraw\w+|"
    r"subsequent(ly)?|thereafter|later|after the other\w*|"
    r"comparator|separately|comparison|"
    r"alternative|unsuitable|first choice|instead|"
    r"not given|not permitted|excluded|withheld|not administered)\b", re.I)

def near_marker(text, chars=60):
    out = []
    for k in (1, 2):
        m = re.search(rf"\[E{k}\].*?\[/E{k}\]", text, re.S)
        if m:
            out.append(text[max(0, m.start() - chars):m.end() + chars])
    return " ".join(out)

def role_adjacency(instances, name):
    c = Counter()
    for r in instances:
        c[(r["label"] != "NONE", bool(ROLE.search(near_marker(r["text"]))))] += 1
    pos = c[(True, True)] / max(c[(True, True)] + c[(True, False)], 1)
    neg = c[(False, True)] / max(c[(False, True)] + c[(False, False)], 1)
    print(f"{name:12s} POS {pos:.3f}  NONE {neg:.3f}  ratio {neg / max(pos, 1e-9):.2f}")
    return neg

print("\nrole language near marker")
role_adjacency(train, "human")          # 0.018 / 0.005, ratio 0.27
role_adjacency(v14, "v14 (0.55)")       # 0.228 / 0.468, ratio 2.05
new_neg = role_adjacency(inst, "v14 (0.20)")
print(f"\nNONE adjacency {0.468:.3f} -> {new_neg:.3f}; corpus is 0.005")


# ---- side effects: the other gates must not have moved -------------------
gates.report(inst, records=generation_records(GEN), strict=False)


# ---- read the zero-assert sentences --------------------------------------
# most non-participants now have no role, so the scene has to carry them.
# the failure to look for is bare lists returning.
shown = 0
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    r = json.loads(line)
    if r.get("error") or r["spec"]["asserts"] or shown >= 10:
        continue
    print(f"[{len(r['spec']['entities'])} ents, {len(r['spec']['roles'])} roles] "
          f"{r['sample']['sentence']}")
    shown += 1